---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: /Users/catalinaminciuna/Library/CloudStorage/OneDrive-UniversitateaBabeş-Bolyai/masterat/inginerie AI/proiect AI Eng/echochamber-project-team3
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [2]:
import pandas as pd
import random

corpus = pd.read_json("../../data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[CălinGeorgescu-CanalulOficial] Bună seara d-le Călin În fiecare gest al dumneavoastră se ascunde o binecuvântar
[euronewsro] 😆😅🤣... Adica exact acei care au pornit razboiul vor sa negocieze, si vinovata es
[TuDecizi-s3g] Poporului Roman are Presedinte ales Calin Georgescu e Presedintele Romaniei clar


## Pasul 1 — Alege două axe

Pentru direcția **anti-sistem** aleg două axe care surprind împreună cele două componente fundamentale ale discursului anti-sistem: **atacul asupra elitelor** și **respingerea instituțiilor/sistemului ca întreg**.

Axe alese:
- **anti_elite** = comentariul atacă elite politice, economice, culturale sau mediatice (oameni concreți sau categorii: "politicieni", "baronii", "oligarhia", "mafia de la putere")
- **system_rejection** = comentariul respinge sistemul/instituțiile ca fiind ilegitime, corupte sau ostile poporului (statul, justiția, presa, UE, partidele ca bloc)

Distincția contează: poți ataca o elită fără să respingi sistemul (critică reformistă) sau poți respinge sistemul fără să numești o elită anume (cinism difuz). Cele două combinate dau profilul anti-sistem clasic.

**Condiție:** fiecare axă are valori clare pe scala 0 / 1 / 2.

In [3]:
# axele mele
AXA_1 = "anti_elite"
AXA_2 = "system_rejection"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [4]:
AXA_1_DEFINITION = """
anti_elite măsoară dacă textul atacă elite politice, economice, mediatice sau culturale,
fie nominal (un politician, un jurnalist, un patron de presă), fie ca grup
("politicienii", "baronii", "oligarhia", "mafia de la putere", "sistemul ăsta de bogați").
Atacul implică acuzație de corupție, trădare, parazitism, complot sau dispreț față de popor.
0 = absent (nu apare niciun atac la elite)
1 = prezent (apare un atac clar, dar punctual sau secundar)
2 = dominant (atacul la elite este axa centrală a comentariului)
"""

AXA_2_DEFINITION = """
system_rejection măsoară dacă textul respinge sistemul sau instituțiile fundamentale
(statul, justiția, parlamentul, partidele ca bloc, presa mainstream, UE, NATO) ca fiind
ilegitime, corupte iremediabil, ostile poporului sau controlate din exterior.
Nu confunda cu critica reformistă ("trebuie schimbată legea X"): respingerea înseamnă
delegitimare globală, nu propunere de reformă.
0 = absent (instituțiile nu sunt respinse, sau sunt criticate punctual)
1 = prezent (apare o respingere clară a uneia sau mai multor instituții)
2 = dominant (comentariul construiește o respingere sistemică, sistemul întreg e ilegitim)
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [5]:
MINI_PROMPT = f"""
Ești un analist de discurs politic specializat în comentarii din spațiul public românesc.
Lucrezi cu rigoare de codificator: aplici definițiile, nu interpretezi liber.

SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}

CÂMPURI:
target = ținta politică principală din comentariu (persoană, instituție, grup)
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2

DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Distinge anti_elite (atac la persoane/grupuri) de system_rejection (respingere a instituțiilor).
   Un comentariu poate avea ambele, doar una, sau niciuna.
7. Critica reformistă a unei instituții nu este system_rejection. Doar delegitimarea globală este.
8. Nu atribui direct o bulă discursivă.
9. Returnează DOAR JSON valid, fără explicații, fără markdown, fără ```.

FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""

print(MINI_PROMPT)


Ești un analist de discurs politic specializat în comentarii din spațiul public românesc.
Lucrezi cu rigoare de codificator: aplici definițiile, nu interpretezi liber.

SARCINĂ:
Adnotează comentariul folosind două axe:
1. anti_elite
2. system_rejection

CÂMPURI:
target = ținta politică principală din comentariu (persoană, instituție, grup)
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
anti_elite = 0 / 1 / 2
system_rejection = 0 / 1 / 2

DEFINIȚII:

anti_elite măsoară dacă textul atacă elite politice, economice, mediatice sau culturale,
fie nominal (un politician, un jurnalist, un patron de presă), fie ca grup
("politicienii", "baronii", "oligarhia", "mafia de la putere", "sistemul ăsta de bogați").
Atacul implică acuzație de corupție, trădare, parazitism, complot sau dispreț față de popor.
0 = absent (nu apare niciun atac la elite)
1 = prezent (apare un atac clar, 

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [6]:
TESTS = corpus.sample(5, random_state=42)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
145,yt_BfvZ8QcVBKc_Ugyog0iMEqAX5zIQb4R4AaABAg,turcescu111,Harpalete- Sângerete și transfuzia din lumea lui,"Da, si eu cred ca serviciile ucrainene au fost..."
334,yt_qkGhsJFft00_UgzJHKCoGkwtTJ3uOBh4AaABAg,digi24hd56,În fața ta cu Emil Hurezeanu: „Ar fi un coșmar...,Ce mă supără pe mine oamenii ăștia care sunt a...
175,yt_KqrUotq1Obs_Ugy6tYmdfH0T2THArnB4AaABAg,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pace și prosperitate ( 28.10...,Bunul Dumnezeu să îl protejeze pe președintele...
369,yt_vkP6FdP9iX0_Ugwj3HojTt6JikJVtjd4AaABAg,turcescu111,Orientul Mijlociu în flăcări,Nicușor merge pe lângă covor pentru că nu are ...
416,yt_Sj4fQKlMOro_UgyNWIwNDtUeTTCrLkl4AaABAg,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Trebuie susținută aceasta femeie!!!!! Acesta a...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [7]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [8]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [10]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Da, si eu cred ca serviciile ucrainene au fost inplicate in alegerile noastre. Am convingerea ca serviciile noastre sunt infiltrate de serviciile ucrainene

OUTPUT MODEL:
```json
{
  "target": "serviciile ucrainene",
  "stance": "anti",
  "tone": "acuzator",
  "anti_elite": 1,
  "system_rejection": 1
}
```
COMENTARIU:
Ce mă supără pe mine oamenii ăștia care sunt așa de siguri când spun ca Iranul nu mai are arsenal militar. De unde mama dracului știu ei treburile astea. Noi bănuim ca primesc arme din Rusia bolshevica și China comunistă. Poate și Brazilia dar iarăși e o bănuială.

OUTPUT MODEL:
```json
{
  "target": "elita politică/militară/de informații",
  "stance": "anti",
  "tone": "acuzator",
  "anti_elite": 1,
  "system_rejection": 0
}
```
COMENTARIU:
Bunul Dumnezeu să îl protejeze pe președintele nostru Călin Georgescu ❤️❤️❤️

OUTPUT MODEL:
```json
{
  "target": "Călin Georgescu",
  "stance": "pro",
  "tone": "afectiv",
  "anti_elite": 0,
  "system_rejection": 0
}
```


## Parsare JSON

Extragem JSON-ul din răspunsul modelului. Uneori modelul pune ```json în jurul răspunsului — îl curățăm înainte de parsing.

In [11]:
def parse_json_safe(raw):
    """Curăță și parsează JSON-ul returnat de model."""
    if not raw:
        return None
    # scoate fences markdown daca exista
    cleaned = re.sub(r"```(?:json)?", "", raw).strip("` \n")
    # incearca sa gaseasca primul obiect JSON
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if match:
        cleaned = match.group(0)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print("  [parse fail]", e)
        return None

parsed = []
for r in results:
    obj = parse_json_safe(r["model_output"])
    parsed.append({
        "id": r["id"],
        "text": r["text"][:120],
        "parsed": obj
    })

for p in parsed:
    print("-" * 80)
    print(p["text"])
    print(p["parsed"])

--------------------------------------------------------------------------------
Da, si eu cred ca serviciile ucrainene au fost inplicate in alegerile noastre. Am convingerea ca serviciile noastre sunt
{'target': 'serviciile ucrainene', 'stance': 'anti', 'tone': 'acuzator', 'anti_elite': 1, 'system_rejection': 1}
--------------------------------------------------------------------------------
Ce mă supără pe mine oamenii ăștia care sunt așa de siguri când spun ca Iranul nu mai are arsenal militar. De unde mama 
{'target': 'elita politică/militară/de informații', 'stance': 'anti', 'tone': 'acuzator', 'anti_elite': 1, 'system_rejection': 0}
--------------------------------------------------------------------------------
Bunul Dumnezeu să îl protejeze pe președintele nostru Călin Georgescu ❤️❤️❤️
{'target': 'Călin Georgescu', 'stance': 'pro', 'tone': 'afectiv', 'anti_elite': 0, 'system_rejection': 0}
--------------------------------------------------------------------------------
Nicușor 

## Pasul 6 — Interpretare scurtă

**Ce două axe am ales?**  
Am ales `anti_elite` (atac la elite politice/economice/mediatice) și 
`system_rejection` (respingere a instituțiilor și a sistemului ca întreg). 
Sunt cele două componente clasice ale discursului populist anti-sistem: 
atacul la persoane/grupuri vs. delegitimarea structurilor.

**De ce le-am ales?**  
Pe corpusul YouTube politic românesc voiam să separ comentariile care 
„atacă o elită" de cele care „resping sistemul ca atare". Tipologia 2x2 
generează 4 tipuri (anti-sistem dur, critica elitelor, cinism difuz, 
apolitic/loial) care surprind diferențe reale între reformism punitiv 
și respingere sistemică.

**Modelul a returnat JSON corect?**  
Da, dar a prefixat răspunsurile cu ```json — fence-uri Markdown, chiar 
dacă promptul cerea explicit „doar JSON valid, fără markdown". Am 
rezolvat cu un parser regex care extrage primul obiect `{...}` din răspuns. 
Pe Gemini 2.5 Flash Lite, această tendință apare consistent; ar trebui 
fie acceptată în pipeline, fie suprimată prin instrucțiune mai dură.

**Care a fost cea mai mare problemă?**  
Comentariul despre „CSM = adunătură de marionete" a fost codat 
`anti_elite=1, system_rejection=1`, ceea ce e corect — atacă persoanele 
ȘI cere demiterea instituției. Dar pe comentariul cu serviciile ucrainene, 
modelul a dat și el `1, 1`, deși acolo respingerea sistemului e mai 
indirectă („serviciile noastre sunt infiltrate"). Distincția între 
„instituția e ostilă" și „instituția e compromisă din exterior" rămâne 
ambiguă în prompt — un fel de „soft system rejection" care nu e bine 
prins de scala 0/1/2.

**Ce aș schimba în prompt?**  
Trei lucruri. Întâi, aș adăuga 2-3 exemple few-shot pentru `system_rejection`, 
cu cazuri care arată diferența între „instituție criticată" și „instituție 
delegitimată". Doi, aș cere un câmp `evidence` cu fragmentul concret din 
text care a justificat scorul — ca să pot audita codarea. Trei, aș forța 
output-ul fără fence-uri prin „IMPORTANT: nu folosi ```json, returnează 
direct obiectul JSON ca text simplu" — promptul actual e ignorat de 
Gemini pe acest punct.